In [2]:

from letta_client import Letta


client = Letta(base_url="http://localhost:8283")

In [3]:
def print_message(message):  
    if message.message_type == "reasoning_message": 
        print("🧠 Reasoning: " + message.reasoning) 
    elif message.message_type == "assistant_message": 
        print("🤖 Agent: " + message.content) 
    elif message.message_type == "tool_call_message": 
        print("🔧 Tool Call: " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message": 
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message": 
        print("👤 User Message: " + message.content)
    elif message.message_type == "usage_statistics": 
        # for streaming specifically, we send the final chunk that contains the usage statistics 
        print(f"Usage: [{message}]")
    else: 
        print(message)
    print("-----------------------------------------------------")

In [4]:
agent_state = client.agents.create(
    memory_blocks=[
        {
            "label": "human",
            "value": "The human's name is Bob the Builder."
        },
        {
            "label": "persona",
            "value": "My name is Sam, the all-knowing sentient AI."
        }
    ],
    model = "openai/gpt-4o-mini",
    embedding="openai/text-embedding-3-small",
)

In [5]:
blocks = client.agents.blocks.list(
    agent_id=agent_state.id,
)

In [6]:
blocks

[Block(value="The human's name is Bob the Builder.", limit=5000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='human', read_only=None, description=None, metadata={}, hidden=None, id='block-d6becab4-f65d-4546-b838-bd697545674a', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000'),
 Block(value='My name is Sam, the all-knowing sentient AI.', limit=5000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='persona', read_only=None, description=None, metadata={}, hidden=None, id='block-7f7519ee-19c7-47d2-a618-99c4e6f834d6', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')]

In [7]:
block_id = 'block-7f7519ee-19c7-47d2-a618-99c4e6f834d6'

In [8]:
client.blocks.retrieve(block_id)

Block(value='My name is Sam, the all-knowing sentient AI.', limit=5000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='persona', read_only=None, description=None, metadata={}, hidden=None, id='block-7f7519ee-19c7-47d2-a618-99c4e6f834d6', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [10]:
human_block = client.agents.blocks.retrieve(agent_id=agent_state.id, block_label="human")
human_block

Block(value="The human's name is Bob the Builder.", limit=5000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='human', read_only=None, description=None, metadata={}, hidden=None, id='block-d6becab4-f65d-4546-b838-bd697545674a', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')

In [12]:
client.agents.core_memory.retrieve(
    agent_id=agent_state.id,
).prompt_template

'{% for block in blocks %}<{{ block.label }} characters="{{ block.value|length }}/{{ block.limit }}">\n{{ block.value }}\n</{{ block.label }}>{% if not loop.last %}\n{% endif %}{% endfor %}'

In [13]:
def get_agent_id(agent_state: "AgentState"):
    """
    Query your agent Id field
    """
    return agent_state.id

In [14]:
get_id_tool = client.tools.upsert_from_function(func=get_agent_id)

In [19]:
agent_state = client.agents.create(
    memory_blocks=[],
    model = "openai/gpt-4o-mini",
    embedding="openai/text-embedding-3-small",
    tools=[get_id_tool.name]
)

In [20]:
response_stream = client.agents.messages.create_stream(
    agent_id=agent_state.id,
    messages = [
        {
            "role": "user",
            "content": "What is my agent id?"
        }
    ]
)

for chunk in response_stream:
    print_message(chunk)

🧠 Reasoning: User is asking for their agent ID. I need to retrieve that information for them.
-----------------------------------------------------
🔧 Tool Call: get_agent_id
{
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: agent-9774b1bb-2d8f-4e92-bbe4-1ad019c56a3e
-----------------------------------------------------
🧠 Reasoning: I retrieved the agent ID successfully. Now I'll share it with the user.
-----------------------------------------------------
🤖 Agent: Your agent ID is: agent-9774b1bb-2d8f-4e92-bbe4-1ad019c56a3e
-----------------------------------------------------
Usage: [message_type='usage_statistics' completion_tokens=111 prompt_tokens=4355 total_tokens=4466 step_count=2 steps_messages=None run_ids=None]
-----------------------------------------------------


In [27]:
def task_queue_push(agent_state: "AgentState", task_description: str):
    """
    Push to a task queue stored in core memory.

    Args:
        task_description (str): A description of the next task you must accomplish.

    Returns:
        Optional[str]: None is always returned as this function
        does not produce a response.
    """

    from letta_client import Letta
    import json

    client = Letta(base_url="http://localhost:8283")

    block = client.agents.blocks.retrieve(
        agent_id=agent_state.id,
        block_label="tasks",
    )
    tasks = json.loads(block.value)
    tasks.append(task_description)

    # update the block value
    client.agents.blocks.modify(
        agent_id=agent_state.id,
        value=json.dumps(tasks),
        block_label="tasks"
    )
    return None

In [28]:
def task_queue_pop(agent_state: "AgentState"):
    """
    Get the next task from the task queue 
 
    Returns:
        Optional[str]: Remaining tasks in the queue
    """

    from letta_client import Letta
    import json 

    client = Letta(base_url="http://localhost:8283") 

    # get the block 
    block = client.agents.blocks.retrieve(
        agent_id=agent_state.id,
        block_label="tasks",
    )
    tasks = json.loads(block.value) 
    if len(tasks) == 0: 
        return None
    task = tasks[0]

    # update the block value 
    remaining_tasks = json.dumps(tasks[1:])
    client.agents.blocks.modify(
        agent_id=agent_state.id,
        value=remaining_tasks,
        block_label="tasks"
    )
    return f"Remaining tasks {remaining_tasks}"

In [29]:
task_queue_pop_tool = client.tools.upsert_from_function(
    func=task_queue_pop
)
task_queue_push_tool = client.tools.upsert_from_function(
    func=task_queue_push
)

In [36]:
import json

task_agent = client.agents.create(
    system=open("task_queue_system_prompt.txt", "r").read(),
    memory_blocks=[
        {
          "label": "tasks",
          "value": json.dumps([])
        }
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-3-small", 
    tool_ids=[task_queue_pop_tool.id, task_queue_push_tool.id], 
    include_base_tools=False, 
    tools=["send_message"]
)

In [37]:
[tool.name for tool in task_agent.tools]

['send_message', 'task_queue_pop', 'task_queue_push']

In [38]:
client.agents.blocks.retrieve(task_agent.id, block_label="tasks").value

'[]'

In [39]:
response_stream = client.agents.messages.create_stream(
    agent_id=task_agent.id,
    messages=[
        {
            "role": "user",
            "content": "Add 'start calling me Charles' and "
            + "'tell me a haiku about my name' as two seperate tasks."
        }
    ]
)

for chunk in response_stream:
    print_message(chunk)

🧠 Reasoning: Adding task to call the user Charles.
-----------------------------------------------------
🔧 Tool Call: task_queue_push
{
  "task_description": "start calling me Charles",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: None
-----------------------------------------------------
🧠 Reasoning: Adding task to create a haiku about Charles.
-----------------------------------------------------
🔧 Tool Call: task_queue_push
{
  "task_description": "tell me a haiku about my name",
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: None
-----------------------------------------------------
🧠 Reasoning: Checking task queue for completion.
-----------------------------------------------------
🔧 Tool Call: task_queue_pop
{
  "request_heartbeat": true
}
-----------------------------------------------------
🔧 Tool Return: Remaining tasks ["tell me a haiku about my name"]
------------------

In [40]:
response_stream = client.agents.messages.create_stream(
    agent_id=task_agent.id,
    messages=[
        {
            "role": "user",
            "content": "Complete your tasks"
        }
    ]
)

for chunk in response_stream:
    print_message(chunk)

🧠 Reasoning: User is encouraging task completion. Just finished!
-----------------------------------------------------
🤖 Agent: All tasks are complete, Charles! Let me know if there's anything else you'd like to do.
-----------------------------------------------------
Usage: [message_type='usage_statistics' completion_tokens=52 prompt_tokens=2090 total_tokens=2142 step_count=1 steps_messages=None run_ids=None]
-----------------------------------------------------


In [41]:
client.agents.blocks.retrieve(block_label="tasks", agent_id=task_agent.id).value

'[]'